# M3: Entrenamiento de Keypoints sobre Ruedas (YOLO11-pose) -> Conteo de Ejes

Ubica un keypoint en el centro de cada rueda visible desde un costado del vehículo.
El número de keypoints visibles con confianza >= 0.35 determina el **número de ejes** para calcular la tarifa de vehículos pesados.

**Métricas generadas para el Informe Técnico:**
- Pose mAP50 y mAP50-95
- MAE (Error Absoluto Medio) en el conteo de ejes y Exactitud exacta de conteo (%)

In [ ]:
!pip install -q ultralytics==8.3.49 roboflow==1.1.49
import os, glob, json, torch
import numpy as np
from roboflow import Roboflow
from ultralytics import YOLO

print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
WORKSPACE = 'tu-workspace-roboflow'
PROJECT_NAME = 'ruedas-ejes-pose'
VERSION_NUM = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT_NAME).version(VERSION_NUM).download('yolov11')
print('Dataset YOLO-pose descargado en:', dataset.location)

In [ ]:
model = YOLO('yolo11s-pose.pt')
model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,
    device=0,
    project='/content/drive/MyDrive/proyecto_peaje/runs_m3',
    name='m3_ruedas_pose',
    exist_ok=True
)

In [ ]:
best_weights = '/content/drive/MyDrive/proyecto_peaje/runs_m3/m3_ruedas_pose/weights/best.pt'
best_model = YOLO(best_weights)
metrics = best_model.val(data=f'{dataset.location}/data.yaml')

# Evaluar error (MAE) y exactitud en el conteo de ejes sobre el split valid
val_images = sorted(glob.glob(f'{dataset.location}/valid/images/*'))
errors = []
exact_hits = 0

for img_path in val_images:
    lbl_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if not os.path.exists(lbl_path):
        continue
    with open(lbl_path) as f:
        line = f.readline().strip().split()
    if len(line) <= 5:
        continue
    # En YOLO-pose: class cx cy w h (x y v) * K
    kpt_vals = [float(v) for v in line[5:]]
    gt_axles = sum(1 for i in range(0, len(kpt_vals), 3) if kpt_vals[i + 2] > 0)

    pred = best_model.predict(img_path, conf=0.25, verbose=False)[0]
    pred_axles = 0
    if pred.keypoints is not None and len(pred.keypoints.data) > 0:
        kpts = pred.keypoints.data[0].cpu().numpy()
        pred_axles = sum(1 for kp in kpts if kp[2] >= 0.35 and (kp[0] > 1 or kp[1] > 1))

    err = abs(gt_axles - pred_axles)
    errors.append(err)
    if err == 0:
        exact_hits += 1

report = {
    'modelo': 'M3_ruedas_pose',
    'pose_mAP50': round(float(metrics.pose.map50), 4),
    'pose_mAP50_95': round(float(metrics.pose.map), 4),
    'box_mAP50': round(float(metrics.box.map50), 4),
    'mae_conteo_ejes': round(float(np.mean(errors)), 4) if errors else None,
    'exactitud_conteo_ejes_pct': round((exact_hits / len(errors)) * 100, 2) if errors else None,
    'muestras_val': len(errors)
}
print(json.dumps(report, indent=2))

os.makedirs('/content/drive/MyDrive/proyecto_peaje/models', exist_ok=True)
!cp /content/drive/MyDrive/proyecto_peaje/runs_m3/m3_ruedas_pose/weights/best.pt /content/drive/MyDrive/proyecto_peaje/models/m3_ruedas_pose.pt
with open('/content/drive/MyDrive/proyecto_peaje/metricas_M3.json', 'w') as f:
    json.dump(report, f, indent=2)